# DLPFC Visium → 模拟双切片 (spatial drift / global DE / spatial static)

本 notebook 逻辑参考 `spatialmeta_tutorial/DLPFC.ipynb`：从真实 Visium 估计 spot 坐标、library size 与基因均值，再按空间函数合成 **A/B 两个条件** 的计数矩阵，并保存为 `adata_A` / `adata_B`。

后半部分演示如何在 **不改动原 SpatialMETA 源码** 的前提下，使用同目录下的封装包 `counterfactual_spatial_cvae` 做预处理与训练（`preset_dlpfc_sim_baseline`）。

**使用前请修改下方 `CONFIG` 中的路径**（Visium 原始目录与输出目录）。

In [1]:
from pathlib import Pat
NOTEBOOK_CWD = Path.cwd()

# ---------- 按需修改 ----------
CONFIG = {
    "sample_id": "151509",
    # Visium：filtered_feature_bc_matrix 所在样本目录
    "visium_root": Path("/home_nfs/sifan.miao/STcode/data/DLPFC/DLPFC_raw"),
    # layer 注释：{sample}_truth.txt
    "truth_root": Path("/home_nfs/sifan.miao/STcode/data/DLPFC/DLPFC_raw"),
    # 模拟 h5ad 输出目录（相对于当前工作目录）
    "output_dir": NOTEBOOK_CWD / "data" / "simulated_dlpfc",
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
print("CWD:", NOTEBOOK_CWD)
print("Output:", CONFIG["output_dir"])

CWD: /home_nfs/sifan.miao/1/method/SpatialMETA-master/counterfactual_spatial_cvae
Output: /home_nfs/sifan.miao/1/method/SpatialMETA-master/counterfactual_spatial_cvae/data/simulated_dlpfc


In [2]:
import warnings

import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")

sample_list = [CONFIG["sample_id"]]
labels = [
    pd.read_csv(
        CONFIG["truth_root"] / sample / f"{sample}_truth.txt",
        sep="\t",
        index_col=0,
        header=None,
        names=["layer"],
    )
    for sample in sample_list
]
adatas = [
    sc.read_visium(
        path=str(CONFIG["visium_root"] / sample),
        count_file="filtered_feature_bc_matrix.h5",
    )
    for sample in sample_list
]

adatas_new = []
for adata, label, name in zip(adatas, labels, sample_list):
    if adata.n_obs == 0:
        continue
    adata.var_names_make_unique()
    label = label.reindex(adata.obs.index)
    adata.obs = adata.obs.join(label, how="left")
    adata.obs["batch"] = name
    adata.layers["count"] = adata.X.copy()
    adata = adata[~adata.obs["layer"].isna()].copy()
    if adata.n_obs == 0:
        continue
    adatas_new.append(adata)

len(adatas_new), adatas_new[0]

(1,
 AnnData object with n_obs × n_vars = 4788 × 33538
     obs: 'in_tissue', 'array_row', 'array_col', 'layer', 'batch'
     var: 'gene_ids', 'feature_types', 'genome'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'count')

In [4]:
import numpy as np
from scipy import sparse


def extract_real_statistics(adata):
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
    coords = adata.obsm["spatial"]
    lib_sizes = X.sum(axis=1)
    gene_means = X.mean(axis=0)
    return coords, lib_sizes, gene_means


def spatial_functions(coords):
    x = coords[:, 0]
    y = coords[:, 1]
    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()
    funcs = {}
    funcs["sin_x"] = np.sin(x)
    funcs["sin_y"] = np.sin(y)
    funcs["radial"] = np.exp(-(x**2 + y**2))
    funcs["checker"] = np.sin(3 * x) * np.sin(3 * y)
    funcs["step_x"] = (x > 0).astype(float) - 0.5
    return funcs


def generate_spatial_drift_slices(
    adata,
    n_drift_genes=300,
    n_global_genes=300,
    n_spatial_static=300,
    alpha=0.8,
    random_state=0,
):
    np.random.seed(random_state)

    adata = adata[adata.obs["in_tissue"] == 1].copy()
    sc.pp.filter_cells(adata, min_counts=500)
    sc.pp.filter_genes(adata, min_cells=50)
    mito_ribo = adata.var_names.str.contains(r"^MT-|^RPL|^RPS", case=False, regex=True)
    adata = adata[:, ~mito_ribo].copy()
    sc.pp.filter_genes(adata, min_cells=int(adata.n_obs * 0.05))

    coords, lib_sizes, gene_means = extract_real_statistics(adata)
    funcs = spatial_functions(coords)
    func_keys = list(funcs.keys())

    n_spots = coords.shape[0]
    n_genes = len(gene_means)
    lib_norm = lib_sizes / lib_sizes.mean()

    all_idx = np.random.permutation(n_genes)
    drift_idx = all_idx[:n_drift_genes]
    global_idx = all_idx[n_drift_genes : n_drift_genes + n_global_genes]
    spatial_static_idx = all_idx[
        n_drift_genes + n_global_genes : n_drift_genes + n_global_genes + n_spatial_static
    ]

    gene_type = {}
    for g in drift_idx:
        gene_type[g] = "spatial_drift"
    for g in global_idx:
        gene_type[g] = "global_DE"
    for g in spatial_static_idx:
        gene_type[g] = "spatial_static"

    X_A = np.zeros((n_spots, n_genes))
    X_B = np.zeros((n_spots, n_genes))

    for g in range(n_genes):
        lam = lib_norm * gene_means[g]
        X_A[:, g] = np.random.poisson(lam)
        X_B[:, g] = np.random.poisson(lam)

    for g in spatial_static_idx:
        mu = gene_means[g]
        f_key = np.random.choice(func_keys)
        f = funcs[f_key]
        f = (f - f.mean()) / f.std()
        lam = np.clip(mu * (1 + alpha * f) * lib_norm, 1e-3, None)
        X_A[:, g] = np.random.poisson(lam)
        X_B[:, g] = np.random.poisson(lam)

    for g in drift_idx:
        mu = gene_means[g]
        fA_key, fB_key = np.random.choice(func_keys, 2, replace=False)
        fA = funcs[fA_key]
        fB = funcs[fB_key]
        fA = (fA - fA.mean()) / fA.std()
        fB = (fB - fB.mean()) / fB.std()
        lam_A = np.clip(mu * (1 + alpha * fA) * lib_norm, 1e-3, None)
        lam_B = np.clip(mu * (1 + alpha * fB) * lib_norm, 1e-3, None)
        X_A[:, g] = np.random.poisson(lam_A)
        X_B[:, g] = np.random.poisson(lam_B)

    for g in global_idx:
        mu = gene_means[g]
        fc = np.random.uniform(2.0, 4.0)
        lam_A = lib_norm * mu
        lam_B = lib_norm * mu * fc
        X_A[:, g] = np.random.poisson(lam_A)
        X_B[:, g] = np.random.poisson(lam_B)

    adata_A = sc.AnnData(X=X_A, obs=adata.obs.copy(), var=adata.var.copy())
    adata_B = sc.AnnData(X=X_B, obs=adata.obs.copy(), var=adata.var.copy())

    for ad in [adata_A, adata_B]:
        ad.obsm["spatial"] = coords
        ad.obs["lib_size"] = lib_sizes
        ad.var["gene_type"] = "background"

    adata_A.obs["slice"] = "A"
    adata_B.obs["slice"] = "B"

    for g, typ in gene_type.items():
        gene_name = adata.var_names[g]
        for ad in [adata_A, adata_B]:
            ad.var.at[gene_name, "gene_type"] = typ

    desp_info = {
        "spatial_drift_genes": drift_idx,
        "global_genes": global_idx,
        "spatial_static_genes": spatial_static_idx,
        "n_drift_genes": n_drift_genes,
        "n_global_genes": n_global_genes,
        "n_spatial_static": n_spatial_static,
        "alpha": alpha,
        "random_state": random_state,
    }

    for ad in [adata_A, adata_B]:
        ad.uns["desp_info"] = desp_info
        ad.layers["count"] = ad.X.copy()
        ad.X = sparse.csr_matrix(ad.X)

    gene_type_by_name = {adata.var_names[idx]: typ for idx, typ in gene_type.items()}
    desp_idx = np.concatenate([drift_idx, global_idx, spatial_static_idx])
    return adata_A, adata_B, desp_idx, gene_type_by_name

In [5]:
adata_A, adata_B, desp_idx, gene_type = generate_spatial_drift_slices(adatas_new[0], random_state=0)

out_a = CONFIG["output_dir"] / f"adata_A_{CONFIG['sample_id']}.h5ad"
out_b = CONFIG["output_dir"] / f"adata_B_{CONFIG['sample_id']}.h5ad"
adata_A.write_h5ad(out_a)
adata_B.write_h5ad(out_b)
print("Saved:", out_a, out_b)
adata_A, adata_B

Saved: /home_nfs/sifan.miao/1/method/SpatialMETA-master/counterfactual_spatial_cvae/data/simulated_dlpfc/adata_A_151509.h5ad /home_nfs/sifan.miao/1/method/SpatialMETA-master/counterfactual_spatial_cvae/data/simulated_dlpfc/adata_B_151509.h5ad


(AnnData object with n_obs × n_vars = 4747 × 6922
     obs: 'in_tissue', 'array_row', 'array_col', 'layer', 'batch', 'n_counts', 'lib_size', 'slice'
     var: 'gene_ids', 'feature_types', 'genome', 'n_cells', 'gene_type'
     uns: 'desp_info'
     obsm: 'spatial'
     layers: 'count',
 AnnData object with n_obs × n_vars = 4747 × 6922
     obs: 'in_tissue', 'array_row', 'array_col', 'layer', 'batch', 'n_counts', 'lib_size', 'slice'
     var: 'gene_ids', 'feature_types', 'genome', 'n_cells', 'gene_type'
     uns: 'desp_info'
     obsm: 'spatial'
     layers: 'count')

In [2]:
import scanpy as sc
adata_A= sc.read_h5ad("/home_nfs/sifan.miao/1/data/DLPFC_diff/151510/adata_A.h5ad")
adata_B= sc.read_h5ad("/home_nfs/sifan.miao/1/data/DLPFC_diff/151510/adata_B.h5ad")

In [4]:
import scanpy as sc
adata= sc.read_h5ad("/home_nfs/sifan.miao/1/data/MOSTA/original/E14.5_E1S2.MOSTA.h5ad")

In [5]:
adata

AnnData object with n_obs × n_vars = 102977 × 28766
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - 4921501E09Rik', 'Regulon - Acaa1a', 'Regulon - Alx1', 'Regulon - Alx3', 'Regulon - Alx4', 'Regulon - Arid3a', 'Regulon - Arnt2', 'Regulon - Arx', 'Regulon - Asap3', 'Regulon - Ascl2', 'Regulon - Atf1', 'Regulon - Atf2', 'Regulon - Atf3', 'Regulon - Atf4', 'Regulon - Atf6', 'Regulon - Atf6b', 'Regulon - Atoh8', 'Regulon - Bach1', 'Regulon - Bach2', 'Regulon - Barhl1', 'Regulon - Barhl2', 'Regulon - Barx1', 'Regulon - Bcl3', 'Regulon - Bcl6', 'Regulon - Bclaf1', 'Regulon - Bdp1', 'Regulon - Bhlha15', 'Regulon - Bhlhe40', 'Regulon - Bmyc', 'Regulon - Boll', 'Regulon - Borcs8', 'Regulon - Bptf', 'Regulon - Brca1', 'Regulon - Brf1', 'Regulon - Brf2', 'Regulon - Carf', 'Regulon - Cbfb', 'Regulon - Ccdc25', 'Regulon - Cdx1', 'Regulon - Cdx2', 'Regulon - Cebpa', 'Regulon - Cebpb', 'Regulon - Cebpd', 'Regulon - Cebpe', 'Regul

In [3]:
adata_A = adata_A[:, adata_A.var['gene_type'].isin(['spatial_drift', 'spatial_static'])]
adata_B = adata_B[:, adata_B.var['gene_type'].isin(['spatial_drift', 'spatial_static'])]

## 使用 `counterfactual_spatial_cvae` 预处理 + 训练

- `prepare_paired_anndata`：写入 `library_size`、`layers['log_norm']`，并对 **A/B 共用** 的坐标做 joint 归一化到约 `[-1,1]`（与 `DLPFC.py` 一致）。
- `preset_dlpfc_sim_baseline()`：`TrainingSchedule.RAMP`，对应原 `fit_core_old` 风格超参。

若包未安装，可在包根目录执行：`pip install -e .`，或将该目录加入 `PYTHONPATH`。

In [6]:
import sys

PKG_ROOT = Path(".").resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from counterfactual_spatial_cvae import (
    CounterfactualSpatialCVAE,
    prepare_paired_anndata,
    preset_dlpfc_sim_baseline,
)

a_prep, b_prep = prepare_paired_anndata(adata_A, adata_B, copy=True, joint_normalize_spatial_coords=True)
preset = preset_dlpfc_sim_baseline()
device = "cuda:1" if __import__("torch").cuda.is_available() else "cpu"
model = CounterfactualSpatialCVAE.from_adata(
    a_prep, b_prep, model_config=preset.model, device=device
)
train_cfg = preset.training
# 演示可改小 epoch；正式实验与 DLPFC.py 一致用 100
train_cfg.max_epoch = 100
train_cfg.verbose = True
history = model.fit(train_cfg)
len(history["epoch_total"]), history["epoch_total"][-1]

Epoch 1/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 2/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 3/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 4/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 5/100:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 97/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 98/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 99/100:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/5 [00:00<?, ?it/s]

(100, 61.81026916503906)

In [7]:
drift_df_2 = model.compute_drift_score(
    a_prep,
    b_prep,
    min_expr_logcpm=0.0,
    top_n_perturb=7000,
    n_grid=20,
    min_spots_per_grid=3,
    alpha=0.4,
    blacklist=True,
    batch_size=1024,
)

[DriftScore] Total genes:         6922
[DriftScore] After expr+blacklist: 6922
[DriftScore] Final genes: 6922
[DriftScore] Top gene: CFL1  score=0.9975  perturb=1.3307  spatial=0.5160  expr_log=2.584

[DriftScore] Statistics by gene type:
                 perturb           spatial_spec               score          
                    mean    median         mean    median      mean    median
gene_type                                                                    
background      0.028629  0.024943     0.424693  0.465404  0.476956  0.479255
global_DE       0.443796  0.338582     0.000016  0.000015  0.590567  0.591231
spatial_drift   0.208778  0.144214     0.496928  0.499584  0.878442  0.913508
spatial_static  0.028314  0.026080     0.442524  0.472123  0.491901  0.493239


In [9]:
def evaluate_spatial_drift_detection(
    result_df,
    adata,
    score_col="score",
    positive_type="spatial_drift",
    gene_type_col="gene_type",
    top_ks=[50,100,200,300,500],
    verbose=True
):
    
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score, average_precision_score

    df = result_df.copy()

    # ---------------------------
    # gene index处理
    # ---------------------------
    if "gene" in df.columns:
        df = df.set_index("gene")

    score_lookup = df[score_col].to_dict()

    all_genes = list(adata.var_names)
    gene_types = adata.var[gene_type_col].to_dict()
    # all_genes = [g for g in all_genes 
    #             if gene_types.get(g) in ('spatial_drift', 'spatial_static')]

    scores = np.array([score_lookup.get(g, 0.0) for g in all_genes])

    # ---------------------------
    # missing score 检查
    # ---------------------------
    missing = [g for g in all_genes if g not in score_lookup]

    if verbose and len(missing) > 0:
        print(f"[Warning] {len(missing)} genes missing score. Filled with 0.")

    # ---------------------------
    # 构建标签
    # ---------------------------
    labels = np.array([
        1 if gene_types.get(g) == positive_type else 0
        for g in all_genes
    ])

    n_pos = int(labels.sum())
    n_total = len(labels)

    if verbose:
        print("Total genes:", n_total)
        print("Positive genes:", n_pos)

    # ---------------------------
    # ranking
    # ---------------------------
    rank_order = np.argsort(-scores)
    sorted_labels = labels[rank_order]

    # ---------------------------
    # AUROC / AUPRC
    # ---------------------------
    try:
        auroc = roc_auc_score(labels, scores)
        auprc = average_precision_score(labels, scores)
    except:
        auroc = 0.0
        auprc = 0.0

    metrics = {
        "AUROC": round(auroc,4),
        "AUPRC": round(auprc,4),
        "n_positive": n_pos
    }

    # ---------------------------
    # Top-k metrics
    # ---------------------------
    for k in top_ks:

        if k > n_total:
            continue

        tp = int(sorted_labels[:k].sum())

        precision = tp / k
        recall = tp / n_pos if n_pos > 0 else 0

        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0

        metrics[f"P@{k}"] = round(precision,4)
        metrics[f"R@{k}"] = round(recall,4)
        metrics[f"F1@{k}"] = round(f1,4)

    metrics_df = pd.DataFrame([metrics])

    # ---------------------------
    # gene type score统计
    # ---------------------------
    type_scores = []

    for g, s in zip(all_genes, scores):

        type_scores.append({
            "gene": g,
            "gene_type": gene_types.get(g,"unknown"),
            "score": s
        })

    type_score_df = pd.DataFrame(type_scores)

    type_score_df = (
        type_score_df
        .groupby("gene_type")["score"]
        .agg(["mean","median","std","max","count"])
        .reset_index()
        .sort_values("mean", ascending=False)
    )

    if verbose:

        print("\n=== Detection Metrics ===")
        print(metrics_df)

        print("\n=== Score Distribution by Gene Type ===")
        print(type_score_df)

    return metrics_df, type_score_df
metrics_df, type_score_df = evaluate_spatial_drift_detection(
    result_df=drift_df_2,
    adata=adata_A,
    score_col="score",
    positive_type="spatial_drift",
    gene_type_col="gene_type",
    top_ks=[5,10,20,30,40,50,100,150,200,250,300],
    verbose=True
)

Total genes: 6922
Positive genes: 300

=== Detection Metrics ===
    AUROC   AUPRC  n_positive  P@5     R@5    F1@5  P@10    R@10   F1@10   
0  0.9617  0.8654         300  1.0  0.0167  0.0328   1.0  0.0333  0.0645  \

   P@20  ...  F1@150  P@200  R@200  F1@200  P@250   R@250  F1@250   P@300   
0   1.0  ...  0.6533  0.945   0.63   0.756   0.88  0.7333     0.8  0.8133  \

    R@300  F1@300  
0  0.8133  0.8133  

[1 rows x 36 columns]

=== Score Distribution by Gene Type ===
        gene_type      mean    median       std       max  count
2   spatial_drift  0.878442  0.913508  0.122097  0.997544    300
1       global_DE  0.590567  0.591231  0.007694  0.606645    300
3  spatial_static  0.491901  0.493239  0.165645  0.932794    300
0      background  0.476956  0.479255  0.169122  0.930685   6022
